# 3-D chromatic shift correction

Measure, validate, save/load, and apply a chromatic shift correction on a
volumetric multi-channel z-stack `(C, Z, Y, X)` using
`ChromaticShiftCorrector3D`.

In [ ]:
import ndv

from microcal import ChromaticShiftCorrector3D, generate_beads_image_3d

## Generate a synthetic 3-D bead volume

The PSF is anisotropic (axially elongated), as in a real z-stack.

In [ ]:
scale_z, scale_y, scale_x = (0.3, 0.1, 0.1)

beads_vol, _ = generate_beads_image_3d(
    n_channels=2,
    shape=(64, 256, 256),
    n_beads=60,
    bead_sigma=(1.7, 1.5, 1.5),
    bead_intensity=60.0,
    bit_depth=16,
    offset=100,
    shifts=[(0, 0, 0), (3.0, 1.5, -2.0)],
    rotations=[0, 3],
    scales=[(1, 1, 1), (1.0, 1.01, 0.99)],
    snr=10,
    seed=42,
)

ndv.imshow(
    beads_vol,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "magenta"}},
    scales={0: 1.0, 1: scale_z, 2: scale_y, 3: scale_x},  # (C, Z, Y, X)
)

## Measure the chromatic shift

`smooth_sigma` / `refine_radius` accept a per-axis `(z, y, x)` tuple for
anisotropy, and an optional `voxel_size` makes bead matching physically
isotropic and reports residuals in physical units.

In [ ]:
csc = ChromaticShiftCorrector3D()
results = csc.measure(
    beads_vol,
    reference_channel=0,
    smooth_sigma=(1.0, 1.5, 1.5),
    min_distance=4,
    threshold_rel=0.3,
    match_max_distance=2,
    min_pairs=2,
    subpixel_refine=True,
    refine_radius=(2, 3, 3),
    voxel_size=(scale_z, scale_y, scale_x),  # optional (z, y, x) in microns
    verbose=True,
)

## Inspect detection and matched pairs

`detection_image` is `(2*C, Z, Y, X)` (volume + sphere mask per channel);
`pairs_image` is a `(Z, Y, X)` label volume.

In [ ]:
ch1_det = results.detection_image[:2]
ndv.imshow(
    ch1_det,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "gray"}},
    scales={0: 1.0, 1: scale_z, 2: scale_y, 3: scale_x},  # (C, Z, Y, X)
)

In [ ]:
ch2_det = results.detection_image[2:4]
ndv.imshow(
    ch2_det,
    channel_mode="composite",
    luts={0: {"cmap": "magenta"}, 1: {"cmap": "gray"}},
    scales={0: 1.0, 1: scale_z, 2: scale_y, 3: scale_x},  # (C, Z, Y, X)
)

In [ ]:
ndv.imshow(
    results.pairs_image.astype("uint16"),
    default_lut={"cmap": "glasbey"},
    scales={0: scale_z, 1: scale_y, 2: scale_x},  # (Z, Y, X)
)

## Validate

Re-detects beads on the corrected volume and reports residuals. With
`voxel_size` set, per-axis (voxel) and physical-unit errors are included.

In [ ]:
val = csc.validate()

## Apply the correction

Here we reuse the bead volume for demonstration.

In [ ]:
vol_corr = csc.apply(image_or_stack=beads_vol, crop=True)
ndv.imshow(
    vol_corr,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "magenta"}},
    scales={0: 1.0, 1: scale_z, 2: scale_y, 3: scale_x},  # (C, Z, Y, X)
)

## Save & load

The JSON stores the 4x4 transforms, the detection params, the `ndim` and
the `voxel_size` — enough to `apply()` later without re-measuring.

In [ ]:
# csc.save("calibration_3d.json")
# csc2 = ChromaticShiftCorrector3D.from_json("calibration_3d.json")
# vol_corr2 = csc2.apply(image_or_stack=beads_vol, crop=True)